# BIFROST: from a chopper calculation to a cascade diagram

`chopcal` works out what BIFROST's six choppers have to do to put a given band on the
sample. `niess` knows where they are. `tof` flies neutrons through them.

This notebook joins the three: pick the band you want, let `chopcal` settle the speeds and
delays, and watch what actually arrives.

`niess.tof` itself needs only `tof`; `chopcal` is this notebook's own requirement, because
it is the thing that knows BIFROST's chopper geometry.

```sh
pip install 'niess[tof]' 'chopcal>=0.5.1'
```

In [ ]:
import chopcal
import scipp as sc

from niess.bifrost import Primary
from niess.instrument import Instrument, Mount
from niess.tof import to_tof_model

## The trade-off

A shorter burst is a sharper pulse and fewer neutrons. Because the settings come from one
number, comparing two is just running `chopcal` twice — the counts below give one half of
the trade, and the overlaid arrival curves give the other.

In [ ]:
ENERGY_MIN = 4.5      # meV, the slowest neutrons you want at the sample
SHAPING_TIME = 2e-4   # s, how long the pulse-shaping choppers stay open

settings = chopcal.bifrost(energy_min=ENERGY_MIN, shaping_time=SHAPING_TIME)
settings

## Handing them to the instrument

`chopcal` names the choppers by function, `niess` by their place on the beam. Each disc
takes a `…speed` and a `…delay` instrument parameter, so the map is one line each.

`chopper.quantities` hands the fields over as scipp scalars carrying their own units, and
`to_tof_model` converts each to whatever the instrument declares — so neither end has to
know what the other chose. That matters here: the table above prints delays in
milliseconds while `chopper.delay` is in seconds, and a scalar cannot be read the wrong
way round.

In [ ]:
NAMES = {
    'ps1': 'pulse_shaping_chopper_1', 'ps2': 'pulse_shaping_chopper_2',
    'fo1': 'frame_overlap_chopper_1', 'fo2': 'frame_overlap_chopper_2',
    'bw1': 'bandwidth_chopper_1',     'bw2': 'bandwidth_chopper_2',
}

def chopper_values(settings):
    values = {}
    for key, name in NAMES.items():
        fields = settings[key].quantities
        values[f'{name}speed'] = fields['speed']
        values[f'{name}delay'] = fields['delay']
    return values

values = chopper_values(settings)
values['bandwidth_chopper_2speed']   # -14 Hz: this one counter-rotates

## Build the model

`Primary.from_calibration()` is BIFROST as surveyed, and an `Instrument` is what holds it:
`origin` names the component everything else is measured to, which is where the last
detector goes. `to_tof_model` walks that tree, so it finds the choppers wherever they sit
— four of BIFROST's six are nested inside guide sections — and measures their distances
*along* the curved guide rather than cutting across it.

In [ ]:
bifrost = Instrument(name='bifrost', origin='sample_origin', parts=(
    Mount(name='primary', content=Primary.from_calibration()),
))

setup = to_tof_model(bifrost, values=values, neutrons=500_000)
setup

Every monitor became a detector, and so did the sample. The table lists every instrument
parameter the model read and what it used — the twelve chopper knobs above are marked
`given`, and anything left at the calibration's own value is marked `default`.

## Fly the neutrons

In [ ]:
result = setup.model.run()
result.plot(visible_rays=5000)

The cascade, read left to right: the pulse-shaping pair cuts a short burst out of the
long ESS pulse, the frame-overlap pair removes what would arrive a frame late, and the
bandwidth pair at 78 m cuts the band down to what the analysers can use.

In [ ]:
for name in setup.detectors:
    counts = int(result.detectors[name].toa.data.sum().value)
    print(f'{name:24s} {counts:8d}')

In [ ]:
result.detectors['sample_origin'].toa.plot()

## What the choppers each did

Each pair, at its own place on the beam.

In [ ]:
(result.choppers['pulse_shaping_chopper_1'].toa.plot()
 + result.choppers['pulse_shaping_chopper_2'].toa.plot())

In [ ]:
(result.choppers['bandwidth_chopper_1'].toa.plot()
 + result.choppers['bandwidth_chopper_2'].toa.plot())

## The trade-off

A shorter burst is a sharper pulse and fewer neutrons. Because the settings come from one
number, comparing two is just running `chopcal` twice.

In [ ]:
comparison = {}
for shaping in (1e-4, 2e-4, 4e-4):
    other = chopcal.bifrost(energy_min=ENERGY_MIN, shaping_time=shaping)
    run = to_tof_model(bifrost, values=chopper_values(other),
                       neutrons=500_000).model.run()
    arriving = run.detectors['sample_origin'].toa
    comparison[f'{shaping * 1e6:.0f} us'] = arriving
    print(f'shaping {shaping * 1e6:5.0f} us -> '
          f'{int(arriving.data.sum().value):7d} at the sample')

In [ ]:
import plopp as pp
pp.plot({name: arriving.data.hist(toa=200) for name, arriving in comparison.items()})

## More than one pulse

Everything so far has been a single pulse, which cannot show what a chopper turning at a
*fraction* of the source frequency is for. `pulses=` simulates several.

The bandwidth pair runs at 14 Hz, the source frequency, so it opens once per pulse and
every pulse gets through. Halve one of them and it opens for every *other* pulse — the
neutrons from the ones in between arrive while it is closed and are absorbed.

In [ ]:
def per_pulse(setup):
    counts = setup.model.run().detectors['sample_origin'].toa.data
    return [int(counts['pulse', i].sum().value) for i in range(counts.sizes['pulse'])]

two_pulses = to_tof_model(bifrost, values=values, pulses=2, neutrons=200_000)
print('bw1 at 14 Hz:', per_pulse(two_pulses))

skipping = two_pulses.with_values(bandwidth_chopper_1speed=sc.scalar(7.0, unit='Hz'))
print('bw1 at  7 Hz:', per_pulse(skipping))

The second pulse is gone entirely — that is pulse skipping, and it is what buys the
wider band a slower disc lets through.

Watch it happen: the second pulse's rays reach the disc at 78 m and stop there.

In [ ]:
skipping.model.run().plot(visible_rays=3000)

## Changing one value

`with_values` rebuilds from the same instrument, as it did above, so a single knob can be
turned without recomputing the rest. Anything overridden is marked `given` in the table,
and everything else keeps the value it had.

In [ ]:
slower = setup.with_values(bandwidth_chopper_1speed=sc.scalar(7.0, unit='Hz'))
slower